# Blind zones, history refresh and content-based ranking: main pipeline

Run the cells in order (Google Colab). Set `ROOT` to the project folder that holds the BitFraud files (`01_data/`) and the outputs (`05_results/bz/`).

## Core functions: data loading, LightGBM expert, metrics, cluster bootstrap

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

"""
Noyau du pipeline zone aveugle (CPU uniquement). Chargement, règle d'historique, strates, expert et bootstrap définis UNE fois.
Protocole figé : D1..D6 ; train D1-D3, validation D4 (early stopping), test D5-D6 ; label_final ; règle r_t = max_a n_H(a).
Deux mémoires d'adresses :
  M_TRAINVAL : D1-D4 -> strates S1/S2/S3 (S2 = adresse vue mais r_t = 0, S3 = jamais vue)
  M_TRAIN    : D1-D3 -> partition adresses vues / nouvelles de la zone aveugle
"""
import json, os, sys, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

SEEDS = [42, 123, 2024, 7, 101, 202]
SNAP = ["D1", "D2", "D3", "D4", "D5", "D6"]
BASE = ["input_count", "output_count", "input_addr_count", "coinbase_flag", "has_witness", "script_type_encoded",
        "input_addr_concentration", "io_count_ratio", "tx_weight", "avg_input_value", "total_input_scaled",
        "log_output_value", "fee_ratio"]
HIST = ["prev_addr_seen_ratio", "prev_addr_seen_count"]
POLICIES = {"static": {"D5": ["D1", "D2", "D3"], "D6": ["D1", "D2", "D3"]},
            "win3": {"D5": ["D2", "D3", "D4"], "D6": ["D3", "D4", "D5"]},
            "all": {"D5": ["D1", "D2", "D3", "D4"], "D6": ["D1", "D2", "D3", "D4", "D5"]}}
LGBM = dict(objective="binary", learning_rate=0.05, num_leaves=63, min_child_samples=20, feature_fraction=0.8,
            bagging_fraction=0.8, bagging_freq=1, reg_lambda=1.0, n_estimators=3000, n_jobs=-1, verbose=-1)


class Data:
    def __init__(self, df, ins, outs, A):
        self.df, self.ins, self.outs, self.A = df, ins, outs, A
        self.snap = df["snapshot_id"].values; self.y = df["label_final"].astype(int).values
        self.edges = pd.concat([ins[["tx_idx", "addr_idx"]], outs[["tx_idx", "addr_idx"]]]).drop_duplicates()
        self.tr = np.where(np.isin(self.snap, SNAP[:3]))[0]; self.va = np.where(self.snap == "D4")[0]
        self.te5 = np.where(self.snap == "D5")[0]; self.te6 = np.where(self.snap == "D6")[0]
        self.te = np.concatenate([self.te5, self.te6]); self.yt = self.y[self.te]; self.sn = self.snap[self.te]
        self.e_te = self.edges[self.edges["tx_idx"].isin(set(self.te.tolist()))]

    def links(self, hist):
        fraud = set(np.where(np.isin(self.snap, hist) & (self.y == 1))[0].tolist())
        fi = self.ins[self.ins["tx_idx"].isin(fraud)]; fo = self.outs[self.outs["tx_idx"].isin(fraud)]
        return np.bincount(fi["addr_idx"].values, minlength=self.A) + np.bincount(fo["addr_idx"].values, minlength=self.A)

    def rule(self, policy):
        parts = []
        for ts, ids in (("D5", self.te5), ("D6", self.te6)):
            tot = self.links(POLICIES[policy][ts]); e = self.e_te[self.e_te["tx_idx"].isin(set(ids.tolist()))]
            parts.append(e.assign(s=tot[e["addr_idx"].values]).groupby("tx_idx")["s"].max().reindex(ids, fill_value=0).values.astype(float))
        return np.concatenate(parts)

    def memory_seen(self, snaps):
        known = np.zeros(self.A, bool)
        known[self.edges[self.edges["tx_idx"].isin(set(np.where(np.isin(self.snap, snaps))[0].tolist()))]["addr_idx"].values] = True
        return self.e_te.assign(h=known[self.e_te["addr_idx"].values]).groupby("tx_idx")["h"].max().reindex(self.te, fill_value=False).values

    def strata(self):
        r = self.rule("static"); seen = self.memory_seen(SNAP[:4])
        return np.where(r > 0, "S1", np.where(seen, "S2", "S3"))

    def dominant_groups(self, policy="all"):
        parts, off = [], 0
        for ts, ids in (("D5", self.te5), ("D6", self.te6)):
            tot = self.links(POLICIES[policy][ts]); e = self.e_te[self.e_te["tx_idx"].isin(set(ids.tolist()))]
            e = e.assign(s=tot[e["addr_idx"].values]).sort_values(["tx_idx", "s"], ascending=[True, False]).drop_duplicates("tx_idx").set_index("tx_idx")
            sv = e["s"].reindex(ids, fill_value=0).values; av = e["addr_idx"].reindex(ids, fill_value=-1).values
            parts.append(np.where(sv > 0, av, self.A + off + np.arange(len(ids)))); off += len(ids)
        return np.concatenate(parts).astype(np.int64)


def load(root=None):
    import sqlite3
    root = root or os.environ.get("ROOT", "/content/drive/MyDrive/bitcoin_snapshots_project")
    db = f"{root}/01_data/databases/all_snapshots_extended_D1_D4_D6_D7_D8_D9_v21.db"
    lab = pd.read_csv(f"{root}/01_data/processed/dataset_v21_full.csv")
    df = lab[lab["snapshot_id"].isin(SNAP)].copy()
    df["rk"] = df["snapshot_id"].map({s: i for i, s in enumerate(SNAP)})
    df = df.sort_values(["rk", "tx_hash"]).reset_index(drop=True).drop(columns="rk")
    num = [c for c in df.columns if df[c].dtype != object]
    df[num] = df[num].replace([np.inf, -np.inf], np.nan).fillna(0)
    h2i = {h: i for i, h in enumerate(df["tx_hash"].values)}
    con = sqlite3.connect(db)
    ins = pd.read_sql_query("SELECT tx_hash, input_address FROM tx_inputs", con)
    outs = pd.read_sql_query("SELECT tx_hash, output_address FROM tx_outputs", con); con.close()
    ins = ins[ins["tx_hash"].isin(h2i)].dropna().copy(); outs = outs[outs["tx_hash"].isin(h2i)].dropna().copy()
    ins["input_address"] = ins["input_address"].astype(str); outs["output_address"] = outs["output_address"].astype(str)
    alla = pd.concat([ins["input_address"], outs["output_address"]]).unique(); a2i = {a: i for i, a in enumerate(alla)}
    ins["tx_idx"] = ins["tx_hash"].map(h2i).astype(np.int64); ins["addr_idx"] = ins["input_address"].map(a2i).astype(np.int64)
    outs["tx_idx"] = outs["tx_hash"].map(h2i).astype(np.int64); outs["addr_idx"] = outs["output_address"].map(a2i).astype(np.int64)
    return Data(df, ins, outs, len(alla))


def fit(Xtr, ytr, Xva, yva, seed):
    import lightgbm as lgb
    m = lgb.LGBMClassifier(random_state=seed, **LGBM)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="average_precision", callbacks=[lgb.early_stopping(150, verbose=False)])
    return m


def expert_probs(D, cols, seeds=SEEDS, cache=None, train_mask=None):
    if cache and os.path.exists(cache) and len(np.load(cache)) == len(D.te):
        return np.load(cache)
    tr = D.tr if train_mask is None else D.tr[train_mask]
    P = np.zeros(len(D.te))
    for sd in seeds:
        m = fit(D.df.iloc[tr][cols], D.y[tr], D.df.iloc[D.va][cols], D.y[D.va], sd)
        P += m.predict_proba(D.df.iloc[D.te][cols])[:, 1] / len(seeds)
    P = np.clip(P, 0, 1 - 1e-9)
    if cache:
        os.makedirs(os.path.dirname(cache), exist_ok=True); np.save(cache, P)
    return P


def metrics(y, s):
    if not 0 < y.sum() < len(y):
        return dict(AUC_ROC=np.nan, AUC_PR=np.nan, lift=np.nan, P_at_R50=np.nan)
    ap = float(average_precision_score(y, s)); p, rc, _ = precision_recall_curve(y, s)
    return dict(AUC_ROC=float(roc_auc_score(y, s)), AUC_PR=ap, lift=ap / y.mean(), P_at_R50=float(p[rc >= 0.5].max()))


class Boot:
    def __init__(self, y, s):
        self.o = np.argsort(-s, kind="stable"); ss = s[self.o]; self.y = y[self.o].astype(np.int64)
        self.st = np.r_[0, np.flatnonzero(np.diff(ss) != 0) + 1]
    def metrics(self, w):
        w = w[self.o]; tp = np.add.reduceat(w * self.y, self.st); fp = np.add.reduceat(w * (1 - self.y), self.st)
        TP = np.cumsum(tp); FP = np.cumsum(fp)
        if TP[-1] == 0 or FP[-1] == 0:
            return float("nan"), float("nan")
        P = TP / np.maximum(TP + FP, 1); R = TP / TP[-1]; ap = float(np.sum(np.diff(R, prepend=0.0) * P))
        x = np.r_[0.0, FP / FP[-1]]; yv = np.r_[0.0, R]
        return ap, float(np.sum(np.diff(x) * (yv[1:] + yv[:-1]) / 2.0))


def cluster_boot(y, scores, gid, pairs, B=1000, seed=0, mask=None):
    idx = np.arange(len(y)) if mask is None else np.where(mask)[0]
    yy = y[idx]; g = np.unique(gid[idx], return_inverse=True)[1]; G = int(g.max()) + 1
    bs = {k: Boot(yy, scores[k][idx]) for k in set(a for p in pairs for a in p)}
    full = {k: v.metrics(np.ones(len(yy))) for k, v in bs.items()}
    rng = np.random.default_rng(seed); acc = {f"{a}-{b}": {"PR": [], "ROC": []} for a, b in pairs}
    for _ in range(B):
        w = np.bincount(rng.integers(0, G, G), minlength=G).astype(float)[g]
        mm = {k: v.metrics(w) for k, v in bs.items()}
        for a, b in pairs:
            acc[f"{a}-{b}"]["PR"].append(mm[a][0] - mm[b][0]); acc[f"{a}-{b}"]["ROC"].append(mm[a][1] - mm[b][1])
    out = {}
    for a, b in pairs:
        k = f"{a}-{b}"
        out[k] = {m: [float(full[a][i] - full[b][i]), float(np.nanpercentile(acc[k][m], 2.5)), float(np.nanpercentile(acc[k][m], 97.5))]
                  for i, m in enumerate(("PR", "ROC"))}
    return out


def save(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(path, index=False)
    else:
        json.dump(obj, open(path, "w"), indent=1)


def synthetic(n=9000, na=700, seed=5):
    rng = np.random.default_rng(seed)
    snap = np.sort(np.array(rng.choice(SNAP, n)))
    df = pd.DataFrame({f: rng.normal(size=n) for f in BASE + HIST}); df["snapshot_id"] = snap
    bad = set(rng.choice(na, 40, replace=False).tolist()); a_in = rng.integers(0, na, n); a_out = rng.integers(0, na, n)
    df["label_final"] = [int(a in bad or c in bad) for a, c in zip(a_in, a_out)]
    ins = pd.DataFrame({"tx_idx": np.arange(n), "addr_idx": a_in}); outs = pd.DataFrame({"tx_idx": np.arange(n), "addr_idx": a_out})
    return Data(df, ins, outs, na)


def _selftest():
    D = synthetic()
    st = D.strata(); r = D.rule("static")
    assert set(st) <= {"S1", "S2", "S3"}

## Strata validation (S1, S2, S3, blind zone) -> 01_validate

In [ ]:
EXPECTED = {"S1": (27260, 5735), "S2": (141926, 138), "S3": (480305, 27), "blind": (622231, 165),
            "seen_M_train": (109575, 135), "new_M_train": (512656, 30)}

def run_validate(D, out=None):
    st = D.strata(); y = D.yt; res = {}
    for k in ("S1", "S2", "S3"):
        res[k] = (int((st == k).sum()), int(y[st == k].sum()))
    b = st != "S1"; res["blind"] = (int(b.sum()), int(y[b].sum()))
    seen = D.memory_seen(SNAP[:3]) & b
    res["seen_M_train"] = (int(seen.sum()), int(y[seen].sum())); res["new_M_train"] = (int((b & ~seen).sum()), int(y[b & ~seen].sum()))
    for pol in ("win3", "all"):
        r = D.rule(pol); res[f"blind_{pol}"] = (int((r == 0).sum()), int(y[r == 0].sum()))
    res["test"] = (int(len(y)), int(y.sum()))
    for k, v in res.items():
        e = EXPECTED.get(k); flag = "" if e is None else ("  OK" if tuple(e) == v else f"  <-- article: {e}")
        print(f"{k:14s} tx {v[0]:>8,d}  positifs {v[1]:>5,d}{flag}")
    if out:
        save({k: list(v) for k, v in res.items()}, f"{out}/validate.json")
    return res

ROOT = "/content/drive/MyDrive/bitcoin_snapshots_project"
D = load(ROOT)
res = run_validate(D, ROOT + "/05_results/bz")

## Rule and history refresh, Table 4 -> 02_rule_refresh

In [ ]:
PAIRS = [("rule_win3", "rule_static"), ("rule_all", "rule_static"), ("rule_win3", "rule_all"),
         ("strict_win3", "rule_win3"), ("strict_static", "rule_static"), ("strict_all", "rule_all")]

def run_refresh(D, P, B=1000, out=None):
    S = {"expert": P}
    for pol in POLICIES:
        r = D.rule(pol); S[f"rule_{pol}"] = r; S[f"strict_{pol}"] = np.where(r > 0, r, P)
    gid = D.dominant_groups("all"); yt, sn = D.yt, D.sn
    periods = (("pooled", np.ones(len(yt), bool)), ("D5", sn == "D5"), ("D6", sn == "D6"))
    t4, bl = [], []
    for pol in POLICIES:
        r = S[f"rule_{pol}"]
        for sub, mk in periods:
            cov = (r > 0) & mk; b = (r == 0) & mk
            t4.append(dict(policy=pol, test=sub, n=int(mk.sum()), pos=int(yt[mk].sum()), covered_tx=float(cov.sum() / mk.sum()),
                           covered_pos=float(yt[cov].sum() / max(1, yt[mk].sum())), blind_tx=int(b.sum()), blind_pos=int(yt[b].sum()),
                           **{f"rule_{k}": v for k, v in metrics(yt[mk], r[mk]).items()}))
            bl.append(dict(policy=pol, test=sub, blind_tx=int(b.sum()), blind_pos=int(yt[b].sum()), **metrics(yt[b], P[b])))
    t4, bl = pd.DataFrame(t4), pd.DataFrame(bl)
    ug, ginv = np.unique(gid, return_inverse=True); gpos = np.bincount(ginv, weights=yt, minlength=len(ug)); gn = np.bincount(ginv, minlength=len(ug))
    order = np.argsort(-gpos); cov_static = np.bincount(ginv, weights=(S["rule_static"] > 0).astype(float), minlength=len(ug))
    groups = pd.DataFrame([dict(rank=i + 1, group_tx=int(gn[k]), group_pos=int(gpos[k]), share_of_positives=float(gpos[k] / yt.sum()),
                                covered_by_static=int(cov_static[k])) for i, k in enumerate(order[:10])])
    drop = {"all": np.zeros(len(ug), bool), "drop_top1": np.isin(np.arange(len(ug)), order[:1]), "drop_top2": np.isin(np.arange(len(ug)), order[:2]),
            "drop_top5": np.isin(np.arange(len(ug)), order[:5]), "drop_ge20": gpos >= 20}
    diff = []
    for vn, d in drop.items():
        keep = ~d[ginv]
        for sub, mk in periods:
            m = keep & mk
            if yt[m].sum() < 2:
                continue
            res = cluster_boot(yt, S, gid, PAIRS, B=B, mask=m)
            for k, v in res.items():
                diff.append(dict(variant=vn, test=sub, pair=k, n_pos=int(yt[m].sum()), dPR=v["PR"][0], PR_lo=v["PR"][1], PR_hi=v["PR"][2],
                                 dROC=v["ROC"][0], ROC_lo=v["ROC"][1], ROC_hi=v["ROC"][2]))
    diff = pd.DataFrame(diff)
    print(t4.round(4).to_string(index=False)); print(groups.to_string(index=False)); print(diff.round(4).to_string(index=False))
    if out:
        for n, x in (("table4", t4), ("blind", bl), ("diff", diff), ("groups", groups)):
            save(x, f"{out}/{n}.csv")
    return t4, bl, diff, groups

P = expert_probs(D, BASE, seeds=SEEDS, cache=f"{ROOT}/05_results/bz/expert_P_base.npy")
t4, bl, diff, groups = run_refresh(D, P, B=1000, out=f"{ROOT}/05_results/bz/02_rule_refresh")

## Expert in the blind zone -> 02_rule_refresh/blind.csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
ROOT = "/content/drive/MyDrive/bitcoin_snapshots_project"
print(pd.read_csv(f"{ROOT}/05_results/bz/02_rule_refresh/blind.csv").round(4).to_string(index=False))

## Load the data

In [ ]:
ROOT = "/content/drive/MyDrive/bitcoin_snapshots_project"
D = load(ROOT)
print(len(D.te), int(D.yt.sum()))   # attendu : 649491 5900

## Anomaly detectors, Table 3 -> 03_detectors

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from scipy.stats import rankdata


def chunked(f, X, n=100000):
    return np.concatenate([f(X[i:i + n]) for i in range(0, len(X), n)])


def run_detectors(D, B=1000, PERM=10000, n_lof=20000, n_svm=10000, seed=0, out=None):
    rng = np.random.default_rng(seed)
    X = D.df[BASE].values.astype(float)
    # entrainement : transactions legitimes D1-D3 sans historique d'adresse anterieur
    tr = D.tr[(D.y[D.tr] == 0) & (D.df["prev_addr_seen_count"].values[D.tr] == 0)]
    sc = StandardScaler().fit(X[tr])
    # evaluation : zone aveugle du test dont aucune adresse n'est dans M_TRAIN (nouvelles adresses)
    reg = (D.strata() != "S1") & ~D.memory_seen(SNAP[:3]); idx = D.te[reg]; y = D.y[idx]; Xte = sc.transform(X[idx])
    print(f"entrainement {len(tr):,d} tx legitimes ; region {len(y):,d} tx, {int(y.sum())} positifs")
    S = {}
    m = IsolationForest(n_estimators=200, random_state=seed, n_jobs=-1).fit(sc.transform(X[tr]))
    S["Isolation Forest"] = -chunked(m.decision_function, Xte)
    sub = rng.choice(tr, min(n_lof, len(tr)), replace=False)
    m = LocalOutlierFactor(n_neighbors=20, novelty=True, n_jobs=-1).fit(sc.transform(X[sub]))
    S["Local Outlier Factor"] = -chunked(m.score_samples, Xte)
    sub = rng.choice(tr, min(n_svm, len(tr)), replace=False)
    m = OneClassSVM(kernel="rbf", gamma="scale", nu=0.05).fit(sc.transform(X[sub]))
    S["One-Class SVM"] = -chunked(m.decision_function, Xte)
    pos = np.where(y == 1)[0]; neg = np.where(y == 0)[0]; N = len(y); n1 = len(pos)
    bs = {k: Boot(y, s) for k, s in S.items()}; res = {k: [] for k in S}
    for _ in range(B):   # bootstrap stratifie : les positifs ET les negatifs sont reechantillonnes
        w = (np.bincount(rng.choice(pos, n1), minlength=N) + np.bincount(rng.choice(neg, len(neg)), minlength=N)).astype(float)
        for k in S:
            res[k].append(bs[k].metrics(w)[1])
    rows = []
    for k, s in S.items():
        r = rankdata(s); obs = (r[pos].sum() - n1 * (n1 + 1) / 2) / (n1 * (N - n1))
        perm = np.array([(r[rng.choice(N, n1, replace=False)].sum() - n1 * (n1 + 1) / 2) / (n1 * (N - n1)) for _ in range(PERM)])
        rows.append(dict(detector=k, AUC_ROC=obs, lo=float(np.nanpercentile(res[k], 2.5)), hi=float(np.nanpercentile(res[k], 97.5)),
                         p_perm=float((1 + (perm >= obs).sum()) / (1 + PERM))))
    t3 = pd.DataFrame(rows); print(t3.round(4).to_string(index=False))
    if out:
        save(t3, f"{out}/table3.csv")
    return t3

t3 = run_detectors(D, out=f"{ROOT}/05_results/bz/03_detectors")

## Fingerprint of the expert -> 04_fingerprint

In [ ]:
def m2(y, s):
    if not 0 < y.sum() < len(y):
        return {}
    o = np.argsort(-s, kind="stable"); ys = y[o]; out = metrics(y, s)
    for k in (5, 10):
        out[f"recall_top{k}pct"] = float(ys[:int(np.ceil(k / 100 * len(y)))].sum() / y.sum())
    return out


def dominant_sets(D):
    gid = D.dominant_groups("all"); yt = D.yt
    ug, ginv = np.unique(gid, return_inverse=True); gpos = np.bincount(ginv, weights=yt, minlength=len(ug)); order = np.argsort(-gpos)
    ag = [(int(ug[k]), float(gpos[k])) for k in order if ug[k] < D.A and gpos[k] > 0]
    tot = D.links(SNAP[:3]); tr_top = {a for a in np.argsort(-tot)[:20].tolist() if tot[a] > 0}
    print("plus grands groupes de positifs du test (adresse, positifs):", ag[:5])
    return {"full": set(), "drop_top2": {a for a, _ in ag[:2]}, "drop_dominant": {a for a, p in ag if p >= 20} | tr_top}


def run_fingerprint(D, ROOT, seeds=SEEDS, B=500, out=None):
    sets = dominant_sets(D); P = {}; rows = []
    R = {pol: D.rule(pol) for pol in ("static", "win3")}
    for v, dom in sets.items():
        bad = set(D.edges[D.edges["addr_idx"].isin(dom)]["tx_idx"].tolist()) if dom else set()
        tr = np.array([i for i in D.tr if i not in bad]); va = np.array([i for i in D.va if i not in bad])
        cache = f"{ROOT}/05_results/bz/fingerprint_P_{v}.npy"
        if v == "full":
            P[v] = expert_probs(D, BASE, seeds=seeds, cache=f"{ROOT}/05_results/bz/expert_P_base.npy")
        elif os.path.exists(cache) and len(np.load(cache)) == len(D.te):
            P[v] = np.load(cache)
        else:
            acc = np.zeros(len(D.te))
            for sd in seeds:
                mdl = fit(D.df.iloc[tr][BASE], D.y[tr], D.df.iloc[va][BASE], D.y[va], sd); acc += mdl.predict_proba(D.df.iloc[D.te][BASE])[:, 1] / len(seeds)
            P[v] = np.clip(acc, 0, 1 - 1e-9); os.makedirs(os.path.dirname(cache), exist_ok=True); np.save(cache, P[v])
        print(f"{v}: {len(dom)} adresses retirees ; entrainement {len(tr):,d} tx, {int(D.y[tr].sum())} positifs")
        for pol in R:
            bl = R[pol] == 0
            rows.append(dict(training=v, policy=pol, train_tx=len(tr), train_pos=int(D.y[tr].sum()), blind_tx=int(bl.sum()), blind_pos=int(D.yt[bl].sum()), **m2(D.yt[bl], P[v][bl])))
    met = pd.DataFrame(rows); print(met.round(4).to_string(index=False))
    rng = np.random.default_rng(0); drows = []
    for pol in R:
        bl = R[pol] == 0; yy = D.yt[bl]; pos = np.where(yy == 1)[0]; neg = np.where(yy == 0)[0]
        bs = {v: Boot(yy, P[v][bl]) for v in sets}; pt = {v: bs[v].metrics(np.ones(len(yy))) for v in sets}; acc = {v: ([], []) for v in sets if v != "full"}
        for _ in range(B):
            w = (np.bincount(rng.choice(pos, len(pos)), minlength=len(yy)) + np.bincount(rng.choice(neg, len(neg)), minlength=len(yy))).astype(float); b0 = bs["full"].metrics(w)
            for v in acc:
                mv = bs[v].metrics(w); acc[v][0].append(mv[0] - b0[0]); acc[v][1].append(mv[1] - b0[1])
        for v, (a, b) in acc.items():
            drows.append(dict(policy=pol, training=v, dPR=pt[v][0] - pt["full"][0], PR_lo=float(np.nanpercentile(a, 2.5)), PR_hi=float(np.nanpercentile(a, 97.5)),
                              dROC=pt[v][1] - pt["full"][1], ROC_lo=float(np.nanpercentile(b, 2.5)), ROC_hi=float(np.nanpercentile(b, 97.5))))
    diff = pd.DataFrame(drows); print(diff.round(4).to_string(index=False))
    if out:
        save(met, f"{out}/fingerprint_metrics.csv"); save(diff, f"{out}/fingerprint_diff.csv")
    return met, diff, P, sets

fp_met, fp_diff, P_fp, DOM = run_fingerprint(D, ROOT, out=f"{ROOT}/05_results/bz/04_fingerprint")

## Consolidation score -> 05_consolidation

In [ ]:
def percentile_profile(ref, pos):
    out = {}
    for c in ref.columns:
        r = np.sort(ref[c].values); v = pos[c].values
        out[c] = float(np.median((np.searchsorted(r, v, side="left") + np.searchsorted(r, v, side="right")) / 2.0 / max(len(r), 1)))
    return pd.Series(out)

def select_features(ref, pos, cols, k=6):
    prof = percentile_profile(ref[cols], pos[cols]); gap = (prof - 0.5).abs().sort_values(ascending=False)
    return {c: (1 if prof[c] > 0.5 else -1) for c in list(gap.index[:k])}, prof

def consolidation_score(X, ref, signs):
    z = np.zeros(len(X))
    for c, s in signs.items():
        sd = ref[c].std(ddof=0); sd = sd if sd > 0 else 1.0; z += s * np.clip((X[c].values - ref[c].mean()) / sd, -5, 5)
    return z / len(signs)

def rankpct(s):
    return pd.Series(s).rank(method="average").values / len(s)


def run_consolidation(D, P, DOM, B=500, out=None):
    df = D.df; sel = np.where(np.isin(D.snap, SNAP[:4]))[0]; iso = df["prev_addr_seen_count"].values == 0
    bad = set(D.edges[D.edges["addr_idx"].isin(DOM["drop_dominant"])]["tx_idx"].tolist()) if DOM["drop_dominant"] else set()
    ok = np.array([i not in bad for i in sel]); sel_nd = sel[ok]
    defs = {"sel_isolated": (sel[(D.y[sel] == 0) & iso[sel]], sel[(D.y[sel] == 1) & iso[sel]]),
            "sel_all": (sel[D.y[sel] == 0], sel[D.y[sel] == 1]),
            "sel_all_nodom": (sel_nd[D.y[sel_nd] == 0], sel_nd[D.y[sel_nd] == 1])}
    Xte = df.iloc[D.te]; S = {"expert": P}; sel_rows = []
    for name, (ri, pi) in defs.items():
        sg, prof = select_features(df.iloc[ri], df.iloc[pi], BASE); print(f"{name}: {len(pi)} positifs, {len(ri)} reference ->", sg)
        sel_rows += [dict(variant=name, feature=c, sign=s, median_pct_train_pos=float(prof[c]), n_pos=len(pi), n_ref=len(ri)) for c, s in sg.items()]
        S[name] = consolidation_score(Xte, df.iloc[ri], sg); S[f"mean_rank(expert,{name})"] = 0.5 * rankpct(P) + 0.5 * rankpct(S[name])
    st = D.strata(); R = {pol: D.rule(pol) for pol in ("static", "win3")}; yt = D.yt; rows = []
    for pol in R:
        bl = R[pol] == 0
        subs = {"blind": bl} if pol != "static" else {"blind": bl, "S2": st == "S2", "S3": st == "S3"}
        for sn, mk in subs.items():
            for v in S:
                rows.append(dict(policy=pol, subset=sn, method=v, n=int(mk.sum()), pos=int(yt[mk].sum()), **m2(yt[mk], S[v][mk])))
    met = pd.DataFrame(rows)
    rng = np.random.default_rng(0); drows = []
    for pol in R:
        bl = R[pol] == 0; yy = yt[bl]; pos = np.where(yy == 1)[0]; neg = np.where(yy == 0)[0]
        bs = {v: Boot(yy, S[v][bl]) for v in S}; pt = {v: bs[v].metrics(np.ones(len(yy))) for v in S}
        res = {v: ([], []) for v in S}; dres = {v: ([], []) for v in S if v != "expert"}
        for _ in range(B):
            w = (np.bincount(rng.choice(pos, len(pos)), minlength=len(yy)) + np.bincount(rng.choice(neg, len(neg)), minlength=len(yy))).astype(float); mm = {v: bs[v].metrics(w) for v in S}
            for v in S:
                res[v][0].append(mm[v][0]); res[v][1].append(mm[v][1])
                if v != "expert":
                    dres[v][0].append(mm[v][0] - mm["expert"][0]); dres[v][1].append(mm[v][1] - mm["expert"][1])
        for v in S:
            r = dict(policy=pol, method=v, AUC_PR=pt[v][0], PR_lo=float(np.nanpercentile(res[v][0], 2.5)), PR_hi=float(np.nanpercentile(res[v][0], 97.5)),
                     AUC_ROC=pt[v][1], ROC_lo=float(np.nanpercentile(res[v][1], 2.5)), ROC_hi=float(np.nanpercentile(res[v][1], 97.5)))
            if v != "expert":
                r.update(dPR=pt[v][0] - pt["expert"][0], dPR_lo=float(np.nanpercentile(dres[v][0], 2.5)), dPR_hi=float(np.nanpercentile(dres[v][0], 97.5)),
                         dROC=pt[v][1] - pt["expert"][1], dROC_lo=float(np.nanpercentile(dres[v][1], 2.5)), dROC_hi=float(np.nanpercentile(dres[v][1], 97.5)))
            drows.append(r)
    diff = pd.DataFrame(drows)
    print(met.round(4).to_string(index=False)); print(diff.round(4).to_string(index=False))
    if out:
        save(met, f"{out}/metrics.csv"); save(diff, f"{out}/diff.csv"); save(pd.DataFrame(sel_rows), f"{out}/selection.csv")
    return met, diff

cons_met, cons_diff = run_consolidation(D, P_fp["full"], DOM, out=f"{ROOT}/05_results/bz/05_consolidation")

## Review workload, Table 5 -> 06_workload

In [ ]:
KS = [0.1, 0.5, 1, 2, 5, 10, 20]

def curve(y, s, rng, B, label):
    n = len(y); P = int(y.sum()); o = np.lexsort((rng.random(n), -s)); ys = y[o].astype(float)
    def at(w):
        cw = np.cumsum(w); cy = np.cumsum(w * ys); out = []
        for k in KS:
            j = min(int(np.searchsorted(cw, np.ceil(k / 100 * cw[-1]))), n - 1); out.append((cy[j] / cy[-1], cy[j] / cw[j]))
        return out
    pt = at(np.ones(n)); pos = np.where(ys == 1)[0]; neg = np.where(ys == 0)[0]; res = np.zeros((B, len(KS), 2))
    for b in range(B):
        w = (np.bincount(rng.choice(pos, len(pos)), minlength=n) + np.bincount(rng.choice(neg, len(neg)), minlength=n)).astype(float); res[b] = at(w)
    rows = []
    for i, k in enumerate(KS):
        lo, hi = np.nanpercentile(res[:, i, 0], [2.5, 97.5])
        rows.append(dict(policy=label, k_pct=k, flagged=int(np.ceil(k / 100 * n)), n_tx=n, n_pos=P, recall=pt[i][0], recall_lo=float(lo), recall_hi=float(hi), precision=pt[i][1]))
    return rows


def run_workload(D, P, B=500, out=None):
    rng = np.random.default_rng(0); rows = []
    for pol in POLICIES:
        bl = D.rule(pol) == 0; rows += curve(D.yt[bl], P[bl], rng, B, pol)
    t5 = pd.DataFrame(rows); print(t5.round(4).to_string(index=False))
    if out:
        save(t5, f"{out}/table5.csv")
    return t5


FIXED = {"input_count": 1, "output_count": -1, "io_count_ratio": -1, "avg_input_value": 1, "log_output_value": 1, "fee_ratio": -1}

def run_fixed(D, P, B=500, out=None):
    df = D.df; sel = np.where(np.isin(D.snap, SNAP[:4]))[0]; iso = df["prev_addr_seen_count"].values == 0
    ref = df.iloc[sel[(D.y[sel] == 0) & iso[sel]]]; s = consolidation_score(df.iloc[D.te], ref, FIXED); st = D.strata(); yt = D.yt; rows = []
    for pol in ("static", "win3"):
        bl = D.rule(pol) == 0
        for sn, mk in ({"blind": bl, "S2": st == "S2", "S3": st == "S3"} if pol == "static" else {"blind": bl}).items():
            rows.append(dict(policy=pol, subset=sn, method="fixed_signature", n=int(mk.sum()), pos=int(yt[mk].sum()), **m2(yt[mk], s[mk])))
    fx = pd.DataFrame(rows); print(fx.round(4).to_string(index=False))
    if out:
        save(fx, f"{out}/fixed_signature.csv")
    return fx

t5 = run_workload(D, P_fp["full"], out=f"{ROOT}/05_results/bz/06_workload")
fx = run_fixed(D, P_fp["full"], out=f"{ROOT}/05_results/bz/06_workload")

## Walk-forward validation, Table 7 -> 07_walkforward

In [ ]:
from scipy.stats import rankdata

def seen_by(D, idx, mem):
    known = np.zeros(D.A, bool)
    known[D.edges[D.edges["tx_idx"].isin(set(np.where(np.isin(D.snap, mem))[0].tolist()))]["addr_idx"].values] = True
    e = D.edges[D.edges["tx_idx"].isin(set(idx.tolist()))]
    return e.assign(h=known[e["addr_idx"].values]).groupby("tx_idx")["h"].max().reindex(idx, fill_value=False).values

WF = [("D1 -> D2", ["D1"], ["D2"]), ("D1D2 -> D3", ["D1", "D2"], ["D3"]),
      ("D1D2D3 -> D4", ["D1", "D2", "D3"], ["D4"]), ("D1D2D3 -> D5D6", ["D1", "D2", "D3"], ["D5", "D6"])]

def fold_scores(D, trs, tes):
    df = D.df; tr = np.where(np.isin(D.snap, trs))[0]; te = np.where(np.isin(D.snap, tes))[0]; iso = df["prev_addr_seen_count"].values == 0
    new = ~seen_by(D, te, trs); idx = te[new]; y = D.y[idx]; out = {}
    for name, (ri, pi) in {"isolated": (tr[(D.y[tr] == 0) & iso[tr]], tr[(D.y[tr] == 1) & iso[tr]]), "all_pos": (tr[D.y[tr] == 0], tr[D.y[tr] == 1])}.items():
        if len(pi) < 5:
            out[name] = None; continue
        sg, _ = select_features(df.iloc[ri], df.iloc[pi], BASE); out[name] = (consolidation_score(df.iloc[idx], df.iloc[ri], sg), sg, len(pi))
    return y, out

def auc_rank(r, pos, N):
    n1 = len(pos); return (r[pos].sum() - n1 * (n1 + 1) / 2) / (n1 * (N - n1))

def run_walkforward(D, B=500, PERM=5000, seed=0, out=None):
    rng = np.random.default_rng(seed); rows = []; pool = {"isolated": [], "all_pos": []}
    for fname, trs, tes in WF:
        y, sc = fold_scores(D, trs, tes); N = len(y); pos = np.where(y == 1)[0]; neg = np.where(y == 0)[0]
        row = dict(fold=fname, new_tx=N, positives=int(y.sum()))
        for v in ("isolated", "all_pos"):
            if sc[v] is None or len(pos) < 2:
                continue
            s = sc[v][0]; r = rankdata(s); obs = auc_rank(r, pos, N)
            perm = np.array([auc_rank(r, rng.choice(N, len(pos), replace=False), N) for _ in range(PERM)])
            b = Boot(y, s); bb = []
            for _ in range(B):
                w = (np.bincount(rng.choice(pos, len(pos)), minlength=N) + np.bincount(rng.choice(neg, len(neg)), minlength=N)).astype(float); bb.append(b.metrics(w)[1])
            row.update({f"AUC_{v}": obs, f"lo_{v}": float(np.nanpercentile(bb, 2.5)), f"hi_{v}": float(np.nanpercentile(bb, 97.5)), f"p_{v}": float((1 + (perm >= obs).sum()) / (1 + PERM))})
            pool[v].append((y, s)); print(f"{fname} {v}: variables {sc[v][1]} ({sc[v][2]} positifs d'entrainement)")
        rows.append(row)
    for v in pool:
        if not pool[v]:
            continue
        y = np.concatenate([a for a, _ in pool[v]]); s = np.concatenate([b for _, b in pool[v]]); r = rankdata(s); N = len(y)
        offs = np.cumsum([0] + [len(a) for a, _ in pool[v]]); npos = [int(a.sum()) for a, _ in pool[v]]
        obs = auc_rank(r, np.where(y == 1)[0], N)
        perm = []
        for _ in range(PERM):
            sel = np.concatenate([offs[i] + rng.choice(offs[i + 1] - offs[i], npos[i], replace=False) for i in range(len(npos))]); perm.append(auc_rank(r, sel, N))
        rows.append({"fold": f"Pooled ({v})", "new_tx": N, "positives": int(y.sum()), f"AUC_{v}": obs, f"p_{v}": float((1 + (np.array(perm) >= obs).sum()) / (1 + PERM))})
    wf = pd.DataFrame(rows); print(wf.round(4).to_string(index=False))
    if out:
        save(wf, f"{out}/table7.csv")
    return wf

wf = run_walkforward(D, out=f"{ROOT}/05_results/bz/07_walkforward")

## Budget routing -> 08_routing

In [ ]:
FEATS15 = BASE + HIST
RF = [("train D1 / val D2 / test D3", ["D1"], ["D2"], ["D3"]), ("train D1D2 / val D3 / test D4", ["D1", "D2"], ["D3"], ["D4"]),
      ("train D1D2D3 / val D4 / test D5D6", ["D1", "D2", "D3"], ["D4"], ["D5", "D6"])]
FGRID = [0, 0.01, 0.02, 0.05, 0.10]; KBUD = [1000, 2000, 5000, 10000]

def lgbm_probs(D, tr, va, targets, seeds):
    P = [np.zeros(len(t)) for t in targets]
    for sd in seeds:
        m = fit(D.df.iloc[tr][FEATS15], D.y[tr], D.df.iloc[va][FEATS15], D.y[va], sd)
        for i, t in enumerate(targets):
            P[i] += m.predict_proba(D.df.iloc[t][FEATS15])[:, 1] / len(seeds)
    return P

def tp_table(y, p, newmask, sc):
    new = np.where(newmask)[0]; seen = np.where(~newmask)[0]
    on = new[np.argsort(-sc[new], kind="stable")]; os_ = seen[np.argsort(-p[seen], kind="stable")]
    cn = np.r_[0, np.cumsum(y[on])]; cs = np.r_[0, np.cumsum(y[os_])]; ca = np.r_[0, np.cumsum(y[np.argsort(-p, kind="stable")])]
    out = {}
    for K in KBUD:
        out[K] = {"baseline": int(ca[min(K, len(y))])}
        for f in FGRID:
            nn = min(int(round(f * K)), len(on)); ns = min(K - nn, len(os_)); out[K][f] = int(cn[nn] + cs[ns])
    return out

def run_routing(D, seeds=SEEDS[:3], out=None):
    iso = D.df["prev_addr_seen_count"].values == 0; res = []
    for name, trs, vas, tes in RF:
        tr = np.where(np.isin(D.snap, trs))[0]; va = np.where(np.isin(D.snap, vas))[0]; te = np.where(np.isin(D.snap, tes))[0]
        sg, _ = select_features(D.df.iloc[tr[(D.y[tr] == 0) & iso[tr]]], D.df.iloc[tr[(D.y[tr] == 1) & iso[tr]]], BASE)
        ref = D.df.iloc[tr[(D.y[tr] == 0) & iso[tr]]]; pv, pt = lgbm_probs(D, tr, va, [va, te], seeds)
        T = {}
        for lab, idx, p in (("val", va, pv), ("test", te, pt)):
            nm = ~seen_by(D, idx, trs); sc = consolidation_score(D.df.iloc[idx], ref, sg)
            T[lab] = tp_table(D.y[idx], p, nm, sc); T[lab + "_new_pos"] = int(D.y[idx][nm].sum()); T[lab + "_new_tx"] = int(nm.sum())
        print(f"{name}: positifs nouvelle adresse val {T['val_new_pos']} / test {T['test_new_pos']} (tx {T['test_new_tx']:,d}) ; variables {sg}")
        res.append((name, T))
    rows = []
    for K in KBUD:
        base = sum(T["test"][K]["baseline"] for _, T in res)
        for proto in ("test_selected_leaky", "validation_selected"):
            chosen, tot = [], 0
            for _, T in res:
                src = T["test"] if proto == "test_selected_leaky" else T["val"]
                best = max(FGRID, key=lambda f: (src[K][f], -f))
                chosen.append(best); tot += T["test"][K][best]
            rows.append(dict(protocol=proto, K=K, f_by_fold=str(chosen), baseline_TP=base, routed_TP=tot, dTP=tot - base))
    rt = pd.DataFrame(rows); print(rt.to_string(index=False))
    full = pd.DataFrame([dict(fold=n, K=K, f=f, TP=T["test"][K][f], baseline=T["test"][K]["baseline"]) for n, T in res for K in KBUD for f in FGRID])
    if out:
        save(rt, f"{out}/table8_9.csv"); save(full, f"{out}/routing_by_f.csv")
    return rt, full

rt, rfull = run_routing(D, out=f"{ROOT}/05_results/bz/08_routing")

## Assemble the result tables

In [ ]:
B = f"{ROOT}/05_results/bz"
rd = lambda p: pd.read_csv(f"{B}/{p}")
f3 = lambda x: f"{x:.3f}"; f2 = lambda x: f"{x:.2f}"; f4 = lambda x: f"{x:.4f}"
def pv(p):
    return "$<0.001$" if p < 0.001 else f"{p:.3f}"
def tab(cols, rows, cap, lab, align=None):
    align = align or "l" + "r" * (len(cols) - 1)
    s = "\\begin{table}[t]\n\\centering\n\\caption{" + cap + "}\n\\label{" + lab + "}\n\\begin{tabular}{" + align + "}\n\\toprule\n" + " & ".join(cols) + " \\\\\n\\midrule\n"
    return s + "\n".join(" & ".join(r) + " \\\\" for r in rows) + "\n\\bottomrule\n\\end{tabular}\n\\end{table}\n"
T = []
t3_ = rd("03_detectors/table3.csv")
T.append(tab(["Detector", "AUC-ROC", "95\\% interval", "$p$"], [[r.detector, f3(r.AUC_ROC), f"[{r.lo:.3f}, {r.hi:.3f}]", pv(r.p_perm)] for r in t3_.itertuples()],
             "Generic anomaly detectors on the new-address region (13 content features, 512,656 transactions, 30 positives). Intervals: stratified bootstrap; $p$: label-permutation test (upper tail).", "tab:detectors"))
t4_ = rd("02_rule_refresh/table4.csv"); nm = {"static": "Static (D1 to D3)", "win3": "Window of 3 snapshots", "all": "Cumulative"}; rows = []
for pol in ("static", "win3", "all"):
    g = t4_[t4_.policy == pol].set_index("test"); rows.append([nm[pol], f3(g.loc["D5", "rule_AUC_PR"]), f3(g.loc["D6", "rule_AUC_PR"]), f3(g.loc["pooled", "rule_AUC_PR"]), f"{int(g.loc['pooled', 'blind_pos'])}"])
T.append(tab(["History policy", "D5", "D6", "Pooled", "Blind positives"], rows, "AUC-PR of the address-history rule by history policy.", "tab:refresh"))
t5_ = rd("06_workload/table5.csv"); rows = []
for k in (1, 5, 10, 20):
    r = [str(k)]
    for pol in ("static", "win3", "all"):
        x = t5_[(t5_.policy == pol) & (t5_.k_pct == k)].iloc[0]; r.append(f"{x.recall:.2f} [{x.recall_lo:.2f}, {x.recall_hi:.2f}]")
    rows.append(r)
n = {p: int(t5_[t5_.policy == p].n_pos.iloc[0]) for p in ("static", "win3", "all")}
T.append(tab(["Examined (\\%)", f"Static ({n['static']} pos.)", f"Window of 3 ({n['win3']})", f"Cumulative ({n['all']})"], rows,
             "Share of blind-zone positives retrieved when the top $k\\%$ of the blind zone is examined, ranked by the expert (95\\% bootstrap interval).", "tab:workload"))
cm = rd("05_consolidation/metrics.csv"); fxx = rd("06_workload/fixed_signature.csv"); cm = pd.concat([cm[cm.policy == "static"], fxx[fxx.policy == "static"]])
names = [("expert", "Expert (supervised)"), ("fixed_signature", "Fixed signature (typology)"), ("sel_isolated", "Selected, no prior history"), ("sel_all", "Selected, all positives"),
         ("sel_all_nodom", "Selected, all positives, no dominant addresses"), ("mean_rank(expert,sel_all)", "Mean rank (expert, selected on all)")]
rows = []
for k, lab in names:
    g = cm[cm.method == k].set_index("subset")
    if "blind" not in g.index:
        continue
    rows.append([lab, f3(g.loc["blind", "AUC_ROC"]), f3(g.loc["S2", "AUC_ROC"]), f3(g.loc["S3", "AUC_ROC"]), f4(g.loc["blind", "AUC_PR"]), f"{g.loc['blind', 'lift']:.1f}", f2(g.loc["blind", "recall_top5pct"])])
T.append(tab(["Method", "ROC Blind", "S2", "S3", "AUC-PR Blind", "Lift", "Recall top 5\\%"], rows, "Content scores in the blind zone under the static history (165 positives).", "tab:scores"))
w = rd("07_walkforward/table7.csv"); rows = []
for r in w.itertuples():
    if str(r.fold).startswith("Pooled (all_pos)"):
        continue
    lab = "Pooled" if str(r.fold).startswith("Pooled") else r.fold.replace("->", "$\\to$")
    rows.append([lab, f"{int(r.positives)}", f3(r.AUC_isolated), pv(r.p_isolated)] if not pd.isna(r.AUC_isolated) else [lab, f"{int(r.positives)}", "n/a", "n/a"])
T.append(tab(["Fold", "Positives", "AUC-ROC", "$p$-value"], rows, "Consolidation score, walk-forward validation across four folds; the features are chosen on the training snapshots of each fold only.", "tab:walkforward"))
r89 = rd("08_routing/table8_9.csv")
def fsel(s):
    v = eval(s); return (f"{v[0]:g} (all folds)" if len(set(v)) == 1 else ", ".join(f"{x:g}" for x in v))
a = r89[r89.protocol == "test_selected_leaky"]; b = r89[r89.protocol == "validation_selected"]
T.append(tab(["Budget $K$", "Baseline TP", "Routed TP", "$\\Delta$TP"], [[f"{int(r.K):,d}", f"{int(r.baseline_TP):,d}", f"{int(r.routed_TP):,d}", f"{int(r.dTP):+d}"] for r in a.itertuples()],
             "Initial routing evaluation, $f$ selected on each fold's test set (3 folds).", "tab:route_leak"))
T.append(tab(["Budget $K$", "$f$ selected", "Baseline TP", "Routed TP", "$\\Delta$TP"], [[f"{int(r.K):,d}", fsel(r.f_by_fold), f"{int(r.baseline_TP):,d}", f"{int(r.routed_TP):,d}", f"{int(r.dTP):+d}"] for r in b.itertuples()],
             "Corrected routing evaluation, $f$ selected on validation and applied unchanged to test (same 3 folds).", "tab:route_valid", "lrrrr"))
tex = "\n".join(T); open(f"{B}/tables.tex", "w").write(tex); print(tex)

## Routing summary

In [ ]:
r = pd.read_csv(f"{ROOT}/05_results/bz/08_routing/routing_by_f.csv")
s = r.groupby(["K", "f"])[["TP", "baseline"]].sum().reset_index(); s["dTP"] = s["TP"] - s["baseline"]
print(s.pivot(index="K", columns="f", values="dTP"))

## Table 2, remedies 1 and 2 (training on the blind regime) -> 11_regime

In [ ]:
def mm(y, s):
    out = metrics(y, s)
    if not 0 < y.sum() < len(y):
        return out
    ys = y[np.argsort(-s, kind="stable")]
    for k in (5, 10):
        out[f"recall_top{k}pct"] = float(ys[:int(np.ceil(k / 100 * len(y)))].sum() / y.sum())
    return out


def strat_boot(y, S, pairs, B=1000, seed=0):
    pos = np.where(y == 1)[0]; neg = np.where(y == 0)[0]
    bs = {k: Boot(y, S[k]) for k in set(a for p in pairs for a in p)}; one = np.ones(len(y))
    full = {k: v.metrics(one) for k, v in bs.items()}; rng = np.random.default_rng(seed)
    acc = {f"{a}-{b}": ([], []) for a, b in pairs}
    for _ in range(B):
        w = (np.bincount(rng.choice(pos, len(pos)), minlength=len(y)) + np.bincount(rng.choice(neg, len(neg)), minlength=len(y))).astype(float)
        m = {k: v.metrics(w) for k, v in bs.items()}
        for a, b in pairs:
            acc[f"{a}-{b}"][0].append(m[a][0] - m[b][0]); acc[f"{a}-{b}"][1].append(m[a][1] - m[b][1])
    out = {}
    for a, b in pairs:
        k = f"{a}-{b}"
        out[k] = {n: [float(full[a][i] - full[b][i]), float(np.nanpercentile(acc[k][i], 2.5)), float(np.nanpercentile(acc[k][i], 97.5))]
                  for i, n in enumerate(("PR", "ROC"))}
    return out


def rule_loo_train(D, tot):
    rows = pd.concat([D.ins[["tx_idx", "addr_idx"]], D.outs[["tx_idx", "addr_idx"]]])
    rows = rows[rows["tx_idx"].isin(set(D.tr.tolist()))]
    mult = rows.groupby(["tx_idx", "addr_idx"]).size().reset_index(name="m")
    mult["s"] = tot[mult["addr_idx"].values] - D.y[mult["tx_idx"].values] * mult["m"].values
    return mult.groupby("tx_idx")["s"].max().reindex(D.tr, fill_value=0).values.astype(float)


def run_regime(D, seeds=SEEDS, B=1000, out=None):
    tot = D.links(SNAP[:3]); r_tr = rule_loo_train(D, tot)
    e_va = D.edges[D.edges["tx_idx"].isin(set(D.va.tolist()))]
    r_va = e_va.assign(s=tot[e_va["addr_idx"].values]).groupby("tx_idx")["s"].max().reindex(D.va, fill_value=0).values
    tr0, va0 = D.tr[r_tr == 0], D.va[r_va == 0]
    print(f"entrainement D1-D3 : {len(D.tr):,d} tx, {int(D.y[D.tr].sum())} positifs ; sans lien historique (r_loo=0) : {len(tr0):,d} tx, {int(D.y[tr0].sum())} positifs")
    print(f"validation D4 : {len(D.va):,d} tx, {int(D.y[D.va].sum())} positifs ; avec r=0 : {len(va0):,d} tx, {int(D.y[va0].sum())} positifs")
    st = D.strata(); blind = st != "S1"; yb = D.yt[blind]
    cfg = {"base_all": (D.tr, D.va), "selV": (D.tr, va0), "trainS": (tr0, D.va), "expert": (tr0, va0)}
    rows, P = [], {c: np.zeros(len(D.te)) for c in cfg}
    for c, (trn, vl) in cfg.items():
        for sd in seeds:
            m = fit(D.df.iloc[trn][BASE], D.y[trn], D.df.iloc[vl][BASE], D.y[vl], sd)
            p = m.predict_proba(D.df.iloc[D.te][BASE])[:, 1]; P[c] += p / len(seeds)
            r = dict(config=c, seed=sd, iters=int(m.best_iteration_ or 0))
            for k, mk in (("blind", blind), ("S2", st == "S2"), ("S3", st == "S3")):
                for n, v in mm(D.yt[mk], p[mk]).items():
                    r[f"{k}_{n}"] = v
            rows.append(r)
        print(f"{c:9s} fini | AUC-ROC zone aveugle (moyenne des graines) {np.mean([r['blind_AUC_ROC'] for r in rows if r['config']==c]):.3f}")
    per = pd.DataFrame(rows); summ = per.drop(columns="seed").groupby("config").agg(["mean", "std"]); summ.columns = [f"{a}_{b}" for a, b in summ.columns]
    cols = [c for c in ["iters_mean", "blind_AUC_ROC_mean", "blind_AUC_ROC_std", "blind_AUC_PR_mean", "blind_lift_mean", "blind_P_at_R50_mean", "S2_AUC_ROC_mean", "S3_AUC_ROC_mean"] if c in summ.columns]
    pd.set_option("display.width", 250); print(summ.loc[list(cfg)][cols].round(4).to_string())
    Pc = {c: np.clip(P[c][blind], 0, 1 - 1e-9) for c in cfg}
    bt = strat_boot(yb, Pc, [("expert", "base_all"), ("selV", "base_all"), ("trainS", "base_all")], B=B)
    for k, v in bt.items():
        print(f"{k} (zone aveugle, graines moyennees) : AUC-ROC {v['ROC'][0]:+.4f} [{v['ROC'][1]:+.4f}, {v['ROC'][2]:+.4f}] ; AUC-PR {v['PR'][0]:+.5f} [{v['PR'][1]:+.5f}, {v['PR'][2]:+.5f}]")
    counts = dict(train_tx=len(D.tr), train_pos=int(D.y[D.tr].sum()), train_r0_tx=int(len(tr0)), train_r0_pos=int(D.y[tr0].sum()),
                  val_tx=len(D.va), val_pos=int(D.y[D.va].sum()), val_r0_tx=int(len(va0)), val_r0_pos=int(D.y[va0].sum()))
    if out:
        save(per, f"{out}/regime_per_seed.csv"); save(summ.reset_index(), f"{out}/regime_summary.csv"); save(dict(boot=bt, counts=counts), f"{out}/regime_boot.json")
    return per, summ, bt, counts

per13, summ13, bt13, counts13 = run_regime(D, out=f"{ROOT}/05_results/bz/11_regime")

## Table 2, remedies 3 and 4 (co-spend, threshold routing) -> 12_remedies

In [ ]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components


def cospend_clusters(D, tx_allowed, maxin):
    e = D.ins[D.ins["tx_idx"].isin(tx_allowed)][["tx_idx", "addr_idx"]].drop_duplicates()
    n_in = e.groupby("tx_idx")["addr_idx"].transform("size"); e = e[(n_in >= 2) & (n_in <= maxin)]
    first = e.groupby("tx_idx")["addr_idx"].transform("first"); a, b = e["addr_idx"].values, first.values; keep = a != b
    g = coo_matrix((np.ones(keep.sum(), dtype=np.int8), (a[keep], b[keep])), shape=(D.A, D.A))
    return connected_components(g, directed=False)[1]


def run_cospend(D, grid_maxin=(5, 20, 100), grid_cap=(1000, None), out=None):
    tot = D.links(SNAP[:3]).astype(float); r_addr = D.rule("static"); a_te = D.e_te["addr_idx"].values
    st = D.strata(); s23 = r_addr == 0; yt = D.yt; prev23 = yt[s23].mean()
    tv = set(np.where(np.isin(D.snap, SNAP[:4]))[0].tolist()); every = set(range(len(D.df)))
    print(f"test {len(D.te):,d} | S1 {int((st=='S1').sum()):,d} | S2 {int((st=='S2').sum()):,d} ({int(yt[st=='S2'].sum())} pos) | S3 {int((st=='S3').sum()):,d} ({int(yt[st=='S3'].sum())} pos) | prevalence S2+S3 {prev23:.5f}")
    base = (average_precision_score(yt, r_addr), roc_auc_score(yt, r_addr)); print(f"regle par adresse : AUC-PR {base[0]:.4f} AUC-ROC {base[1]:.4f}")
    rows = []
    for gname, allowed in (("causal", tv), ("transductive", every)):
        for maxin in grid_maxin:
            comp = cospend_clusters(D, allowed, maxin); size = np.bincount(comp); cl_tot = np.bincount(comp, weights=tot)
            for cap in grid_cap:
                ok = np.ones(D.A, bool) if cap is None else (size[comp] <= cap)
                eff = np.where(ok, cl_tot[comp], tot)
                r_cl = D.e_te.assign(s=eff[a_te]).groupby("tx_idx")["s"].max().reindex(D.te, fill_value=0).values
                comb = r_addr + 1e-3 * r_cl; hit = s23 & (r_cl > 0)
                row = dict(graph=gname, maxin=maxin, cap=cap if cap else "none", max_cluster=int(size.max()),
                           AUC_PR_comb=float(average_precision_score(yt, comb)), AUC_ROC_comb=float(roc_auc_score(yt, comb)),
                           hit_n=int(hit.sum()), hit_pos=int(yt[hit].sum()), hit_precision=float(yt[hit].mean()) if hit.sum() else np.nan,
                           hit_lift=float(yt[hit].mean() / prev23) if hit.sum() else np.nan, hit_legit=int((yt[hit] == 0).sum()),
                           recovered_S2=int(yt[hit & (st == "S2")].sum()), recovered_S3=int(yt[hit & (st == "S3")].sum()))
                rows.append(row)
                print(f"{gname:12s} maxin {maxin:<4d} cap {str(cap):<5s} | hits {row['hit_n']:>7,d} pos {row['hit_pos']:>3d} prec {row['hit_precision']:.4f} lift {row['hit_lift']:.1f} | S2 {row['recovered_S2']} S3 {row['recovered_S3']} | AUC-PR comb {row['AUC_PR_comb']:.4f} (adresse {base[0]:.4f})")
    res = pd.DataFrame(rows)
    if out:
        save(res, f"{out}/cospend_grid.csv")
    return res


TAUS = [1, 2, 3, 5, 10, 20, np.inf]


def routed(r, p, tau):
    return np.where(r >= tau, r + 1.0, p)


def run_tau(D, P, B=300, out=None):
    R = {pol: D.rule(pol) for pol in POLICIES}; gid = D.dominant_groups("all"); yt, sn = D.yt, D.sn
    ug, ginv = np.unique(gid, return_inverse=True); gpos = np.bincount(ginv, weights=yt, minlength=len(ug))
    variants = {"all": np.ones(len(yt), bool), "drop_ge20": ~(gpos >= 20)[ginv]}
    tests = {"pooled": np.ones(len(yt), bool), "D5": sn == "D5", "D6": sn == "D6"}; rows = []
    for vn, vm in variants.items():
        for tn, tm in tests.items():
            mk = vm & tm
            if yt[mk].sum() < 2:
                continue
            for pol in POLICIES:
                for tau in TAUS:
                    r = metrics(yt[mk], routed(R[pol], P, tau)[mk])
                    rows.append(dict(variant=vn, test=tn, policy=pol, tau=("inf" if np.isinf(tau) else int(tau)), n_pos=int(yt[mk].sum()), AUC_PR=r["AUC_PR"], AUC_ROC=r["AUC_ROC"]))
    tab = pd.DataFrame(rows)
    for (vn, pol), d in tab.groupby(["variant", "policy"]):
        print(f"\nAUC-PR | variante {vn} | politique {pol}\n", d.pivot_table(index="test", columns="tau", values="AUC_PR", sort=False).round(4).to_string())
    print("\ntau choisi sur D5 (variante all, meilleur AUC-PR), evalue sur D6 :")
    sel = {}
    for pol in POLICIES:
        d5 = tab[(tab.variant == "all") & (tab.test == "D5") & (tab.policy == pol)].dropna(subset=["AUC_PR"]); best = d5.loc[d5.AUC_PR.idxmax()]; t = best.tau
        d6 = tab[(tab.variant == "all") & (tab.test == "D6") & (tab.policy == pol)].set_index("tau")
        sel[pol] = dict(tau_star=str(t), D5=float(best.AUC_PR), D6_at_star=float(d6.loc[t, "AUC_PR"]), D6_tau1=float(d6.loc[1, "AUC_PR"]), D6_inf=float(d6.loc["inf", "AUC_PR"]))
        print(f"  {pol:7s} tau*={t}  D5 {best.AUC_PR:.4f} | D6 a tau* {sel[pol]['D6_at_star']:.4f} (tau=1 : {sel[pol]['D6_tau1']:.4f}, tau=inf : {sel[pol]['D6_inf']:.4f})")
    diffs = []
    S = {f"tau{('inf' if np.isinf(t) else int(t))}": routed(R["win3"], P, t) for t in TAUS}; pairs = [(k, "tau1") for k in S if k != "tau1"]
    for vn, vm in variants.items():
        for tn, tm in tests.items():
            mk = vm & tm
            if yt[mk].sum() < 2:
                continue
            res = cluster_boot(yt, S, gid, pairs, B=B, mask=mk)
            for k, v in res.items():
                diffs.append(dict(variant=vn, test=tn, pair=k, dPR=v["PR"][0], lo=v["PR"][1], hi=v["PR"][2]))
    diff = pd.DataFrame(diffs); print("\npolitique win3 : AUC-PR(tau) - AUC-PR(tau=1), bootstrap par groupes, IC 95 % :\n", diff.round(4).to_string(index=False))
    if out:
        save(tab, f"{out}/tau_table.csv"); save(diff, f"{out}/tau_diff.csv"); save(sel, f"{out}/tau_selected.json")
    return tab, diff, sel

OUT14 = f"{ROOT}/05_results/bz/12_remedies"
cos = run_cospend(D, out=OUT14)
P = expert_probs(D, BASE, cache=f"{ROOT}/05_results/bz/expert_P_base.npy")
tau_tab, tau_diff, tau_sel = run_tau(D, P, out=OUT14)

## Section 4.1 (strata comparison) and profile of the positives -> 13_strata_profile

In [ ]:
from scipy.stats import ks_2samp
F15 = BASE + HIST


def run_strata_models(D, seeds=SEEDS, out=None):
    st = D.strata(); yt = D.yt; r = D.rule("static")
    sets = {"lgbm_avec_memoire": F15, "lgbm_sans_memoire": BASE}; per, Pm = [], {}
    for name, cols in sets.items():
        acc = np.zeros(len(D.te))
        for sd in seeds:
            m = fit(D.df.iloc[D.tr][cols], D.y[D.tr], D.df.iloc[D.va][cols], D.y[D.va], sd); p = m.predict_proba(D.df.iloc[D.te][cols])[:, 1]; acc += p / len(seeds)
            for k, mk in (("S1", st == "S1"), ("S2", st == "S2"), ("S3", st == "S3"), ("blind", st != "S1"), ("all", np.ones(len(yt), bool))):
                per.append(dict(model=name, seed=sd, stratum=k, **mm(yt[mk], p[mk])))
        Pm[name] = acc; print(f"{name} fini")
    rows = []
    for k, mk in (("S1", st == "S1"), ("S2", st == "S2"), ("S3", st == "S3"), ("blind", st != "S1"), ("all", np.ones(len(yt), bool))):
        rows.append(dict(model="regle", seed=-1, stratum=k, **mm(yt[mk], r[mk])))
    per = pd.DataFrame(per + rows); summ = per.drop(columns="seed").groupby(["model", "stratum"]).agg(["mean", "std"]); summ.columns = [f"{a}_{b}" for a, b in summ.columns]
    cols = [c for c in ["AUC_PR_mean", "AUC_PR_std", "AUC_ROC_mean", "lift_mean", "P_at_R50_mean"] if c in summ.columns]
    pd.set_option("display.width", 250); print(summ[cols].round(4).to_string())
    for name in Pm:
        print(name, {k: round(mm(yt[mk], Pm[name][mk]).get("AUC_PR", np.nan), 4) for k, mk in (("S1", st == "S1"), ("S2", st == "S2"), ("S3", st == "S3"))})
    if out:
        save(per, f"{out}/strata_per_seed.csv"); save(summ.reset_index(), f"{out}/strata_summary.csv")
    return per, summ


def run_profile(D, out=None):
    new = ~D.memory_seen(SNAP[:3]); y = D.yt; X = D.df.iloc[D.te][BASE].values
    print(f"region a adresses nouvelles (M_TRAIN) : {int(new.sum()):,d} tx, {int(y[new].sum())} positifs, {int(((y==0)&new).sum()):,d} legitimes")
    rows = []
    for j, f in enumerate(BASE):
        pos = X[new & (y == 1), j]; neg = X[new & (y == 0), j]
        if len(pos) < 2:
            continue
        ks = ks_2samp(pos, neg); sneg = np.sort(neg)
        pr = 100 * (np.searchsorted(sneg, pos, "left") + np.searchsorted(sneg, pos, "right")) / (2 * len(sneg))
        rows.append(dict(feature=f, KS=float(ks.statistic), p_value=float(ks.pvalue), median_percentile=float(np.median(pr)), mean_percentile=float(pr.mean())))
    if not rows:
        print("moins de 2 positifs dans la region : profil impossible"); return None
    tab = pd.DataFrame(rows).sort_values("p_value"); tab["signif_p<0.001"] = tab["p_value"] < 1e-3
    pd.set_option("display.width", 250); print(tab.round(4).to_string(index=False))
    if out:
        save(tab, f"{out}/profile_ks.csv")
    return tab

OUT15 = f"{ROOT}/05_results/bz/13_strata_profile"
per15, summ15 = run_strata_models(D, out=OUT15)
prof = run_profile(D, out=OUT15)

## Table 2, remedy 5 (enriched features) -> 14_enriched

In [ ]:
EXTRA_CSV = ["fee_per_byte", "fee_log", "fee_per_input", "fee_per_output", "fee_urgency_ratio"]
TIME = ["b_hour", "b_dow"]


def run_enriched(D, ROOT, seeds=SEEDS, B=500, out=None):
    pk = f"{ROOT}/05_results/behavior/behavior_features.pkl"
    if not os.path.exists(pk):
        print("behavior_features.pkl introuvable :", pk, "-> remede 5 non recalcule (a retirer ou a reformuler)"); return None
    Fb = pd.read_pickle(pk)
    assert len(Fb) == len(D.df), f"longueurs differentes : {len(Fb)} contre {len(D.df)}"
    c = np.corrcoef(Fb["b_n_out_rows"].values, D.df["output_count"].values)[0, 1]
    print(f"{Fb.shape[1]} variables comportementales ; controle d'alignement corr(b_n_out_rows, output_count) = {c:.3f}")
    assert c > 0.8, "alignement douteux entre le fichier de variables et la table des transactions"
    X = pd.concat([D.df.reset_index(drop=True), Fb.reset_index(drop=True)], axis=1); X = X.loc[:, ~X.columns.duplicated()]
    newb = [k for k in Fb.columns if k.startswith("b_") and k not in TIME]
    var = {"base": BASE, "rich": BASE + EXTRA_CSV + newb, "rich_time": BASE + EXTRA_CSV + newb + TIME}
    P = {}
    for name, cols in var.items():
        acc = np.zeros(len(D.te))
        for sd in seeds:
            m = fit(X.iloc[D.tr][cols], D.y[D.tr], X.iloc[D.va][cols], D.y[D.va], sd); acc += m.predict_proba(X.iloc[D.te][cols])[:, 1] / len(seeds)
        P[name] = np.clip(acc, 0, 1 - 1e-9); print(f"{name} : {len(cols)} variables, fini")
    rows = []
    for pol in POLICIES:
        bl = D.rule(pol) == 0
        for v in P:
            rows.append(dict(variant=v, policy=pol, blind_tx=int(bl.sum()), blind_pos=int(D.yt[bl].sum()), **mm(D.yt[bl], P[v][bl])))
    blind = pd.DataFrame(rows); pd.set_option("display.width", 250); print(blind.round(4).to_string(index=False))
    drows = []
    for pol in ("static", "win3"):
        bl = D.rule(pol) == 0
        res = strat_boot(D.yt[bl], {v: P[v][bl] for v in P}, [("rich", "base"), ("rich_time", "rich")], B=B)
        for k, v in res.items():
            drows.append(dict(policy=pol, pair=k, dPR=v["PR"][0], PR_lo=v["PR"][1], PR_hi=v["PR"][2], dROC=v["ROC"][0], ROC_lo=v["ROC"][1], ROC_hi=v["ROC"][2]))
    diff = pd.DataFrame(drows); print(diff.round(4).to_string(index=False))
    if out:
        save(blind, f"{out}/enriched_blind.csv"); save(diff, f"{out}/enriched_diff.csv")
    return blind, diff

enr = run_enriched(D, ROOT, out=f"{ROOT}/05_results/bz/14_enriched")